In [171]:
import re
import pandas as pd

# #the script to load the large file and filter by mutation type is below. the ouput is the filtered file that is easier to run

# #Load large ClinVar file
# df = pd.read_csv("data/variant_summary.txt", sep="\t", low_memory=False)
# # variant summary txt file was downloaded from aplpafold

# # Filter for BRCA1 missense SNVs with valid pathogenicity
# df_brca1 = df[
#     (df["GeneSymbol"] == "BRCA1") &
#     (df["Type"] == "single nucleotide variant") &
#     (df["ClinicalSignificance"].notna()) &
#     (df["Name"].str.contains("p.", na=False))
# ]

# # Optional: keep only relevant columns
# columns_to_keep = [
#     "GeneSymbol", "Type", "ClinicalSignificance", "Name", "ReviewStatus", "Assembly", "Chromosome", "Start", "Stop"
# ]
# df_brca1 = df_brca1[columns_to_keep]

# # Save result
# df_brca1.to_csv("data/clinvar_brca1_filtered.csv", index=False)

# print(f"Filtered data saved with {len(df_brca1)} rows.")


In [172]:
df = pd.read_csv("data/clinvar_brca1_filtered.csv") #loading the filtered file

In [173]:
# Extract mutation using regex pattern
def extract_protein_change(name):
    match = re.search(r"\(p\.([A-Za-z]{3})(\d+)([A-Za-z]{3})\)", name)
    if match:
        return match.group(0), match.group(1), match.group(2), match.group(3)
    return None, None, None, None

df[["protein_change", "wt_aa_3", "position", "mut_aa_3"]] = df["Name"].apply(
    lambda name: pd.Series(extract_protein_change(name))
)
df = df.dropna(subset=["wt_aa_3", "mut_aa_3", "position"])

In [174]:
#print(df.columns)

Index(['GeneSymbol', 'Type', 'ClinicalSignificance', 'Name', 'ReviewStatus',
       'Assembly', 'Chromosome', 'Start', 'Stop', 'protein_change', 'wt_aa_3',
       'position', 'mut_aa_3'],
      dtype='object')


In [176]:
def encode_significance(value):
    if "Pathogenic" in value:
        return 1
    elif "Benign" in value:
        return 0
    else:
        return None  # optionally drop VUS and other uncertain cases

In [177]:
df["label"] = df["ClinicalSignificance"].apply(encode_significance)

In [178]:
df = df[df["label"].notna()]

In [179]:
aa_3_to_1 = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
    'Ter': '*'
}

df["wt_aa"] = df["wt_aa_3"].map(aa_3_to_1)
df["mut_aa"] = df["mut_aa_3"].map(aa_3_to_1)

In [180]:
df.to_csv("data/brca1_parsed_mutations.csv", index=False)

In [181]:
aa_properties = {
    # dictionary of amino acid properties
    'A': {'hydrophobicity': 1.8,  'charge':  0, 'polarity': 8.1, 'mw': 89.1,  'volume': 88.6},
    'R': {'hydrophobicity': -4.5, 'charge': +1, 'polarity': 10.5, 'mw': 174.2, 'volume': 173.4},
    'N': {'hydrophobicity': -3.5, 'charge':  0, 'polarity': 11.6, 'mw': 132.1, 'volume': 114.1},
    'D': {'hydrophobicity': -3.5, 'charge': -1, 'polarity': 13.0, 'mw': 133.1, 'volume': 111.1},
    'C': {'hydrophobicity': 2.5,  'charge':  0, 'polarity': 5.5, 'mw': 121.2, 'volume': 108.5},
    'Q': {'hydrophobicity': -3.5, 'charge':  0, 'polarity': 10.5, 'mw': 146.2, 'volume': 143.8},
    'E': {'hydrophobicity': -3.5, 'charge': -1, 'polarity': 12.3, 'mw': 147.1, 'volume': 138.4},
    'G': {'hydrophobicity': -0.4, 'charge':  0, 'polarity': 9.0, 'mw': 75.1,  'volume': 60.1},
    'H': {'hydrophobicity': -3.2, 'charge': +0.5, 'polarity': 10.4, 'mw': 155.2, 'volume': 153.2},
    'I': {'hydrophobicity': 4.5,  'charge':  0, 'polarity': 5.2, 'mw': 131.2, 'volume': 166.7},
    'L': {'hydrophobicity': 3.8,  'charge':  0, 'polarity': 4.9, 'mw': 131.2, 'volume': 166.7},
    'K': {'hydrophobicity': -3.9, 'charge': +1, 'polarity': 11.3, 'mw': 146.2, 'volume': 168.6},
    'M': {'hydrophobicity': 1.9,  'charge':  0, 'polarity': 5.7, 'mw': 149.2, 'volume': 162.9},
    'F': {'hydrophobicity': 2.8,  'charge':  0, 'polarity': 5.2, 'mw': 165.2, 'volume': 189.9},
    'P': {'hydrophobicity': -1.6, 'charge':  0, 'polarity': 8.0, 'mw': 115.1, 'volume': 112.7},
    'S': {'hydrophobicity': -0.8, 'charge':  0, 'polarity': 9.2, 'mw': 105.1, 'volume': 89.0},
    'T': {'hydrophobicity': -0.7, 'charge':  0, 'polarity': 8.6, 'mw': 119.1, 'volume': 116.1},
    'W': {'hydrophobicity': -0.9, 'charge':  0, 'polarity': 5.4, 'mw': 204.2, 'volume': 227.8},
    'Y': {'hydrophobicity': -1.3, 'charge':  0, 'polarity': 6.2, 'mw': 181.2, 'volume': 193.6},
    'V': {'hydrophobicity': 4.2,  'charge':  0, 'polarity': 5.9, 'mw': 117.1, 'volume': 140.0}
}

In [182]:
def compute_property_diff(row, prop):
    wt = row['wt_aa']
    mut = row['mut_aa']
    if wt not in aa_properties or mut not in aa_properties:
        return None  # or np.nan
    return aa_properties[mut][prop] - aa_properties[wt][prop]

for prop in ['hydrophobicity', 'charge', 'polarity', 'mw', 'volume']:
    df[f'delta_{prop}'] = df.apply(lambda row: compute_property_diff(row, prop), axis=1)

In [183]:
df[['wt_aa', 'mut_aa', 'delta_hydrophobicity', 'delta_charge', 'delta_polarity']].head()

,wt_aa,mut_aa,delta_hydrophobicity,delta_charge,delta_polarity
0,C,G,-2.9,0.0,3.5
1,C,G,-2.9,0.0,3.5
2,C,G,-2.9,0.0,3.5
3,C,G,-2.9,0.0,3.5
4,S,N,-2.7,0.0,2.4


In [184]:
# Check if any rows have missing (NaN) values
df[['delta_hydrophobicity', 'delta_charge', 'delta_polarity']].isnull().sum()

delta_hydrophobicity    1092
delta_charge            1092
delta_polarity          1092
dtype: int64

In [185]:
df.to_csv("data/brca1_with_biochemical_features.csv", index=False)

In [186]:
#df[df.isnull().any(axis=1)].head()

In [187]:
df = df.dropna(subset=[
    'delta_hydrophobicity', 'delta_charge',
    'delta_polarity', 'delta_mw', 'delta_volume'
])

In [188]:
dropped = df[df[['delta_hydrophobicity', 'delta_charge']].isnull().any(axis=1)]
dropped.to_csv("data/dropped_due_to_missing_aa.csv", index=False)

In [189]:
domains = {
    "RING": (24, 65),        # Example from UniProt
    "BRCT1": (1642, 1736),
    "BRCT2": (1760, 1855)
}

In [191]:
#df = df.loc[:, ~df.columns.duplicated()] # had two columns named "position", this code removed duplicate columns

In [192]:
df['position'] = df['position'].astype(int)
# did this because position number were floats and not integers

In [193]:
def is_in_domain(pos, start, end):
    return int(start <= pos <= end)

for domain_name, (start, end) in domains.items():
    df[f'in_{domain_name}_domain'] = df['position'].apply(lambda x: is_in_domain(x, start, end))

In [194]:
df[['position', 'in_RING_domain', 'in_BRCT1_domain', 'in_BRCT2_domain']].head()

,position,in_RING_domain,in_BRCT1_domain,in_BRCT2_domain
0,64,1,0,0
1,64,1,0,0
2,61,1,0,0
3,61,1,0,0
4,1040,0,0,0


In [195]:
print(df.columns.tolist())

['GeneSymbol', 'Type', 'ClinicalSignificance', 'Name', 'ReviewStatus', 'Assembly', 'Chromosome', 'Start', 'Stop', 'protein_change', 'wt_aa_3', 'position', 'mut_aa_3', 'label', 'wt_aa', 'mut_aa', 'delta_hydrophobicity', 'delta_charge', 'delta_polarity', 'delta_mw', 'delta_volume', 'in_RING_domain', 'in_BRCT1_domain', 'in_BRCT2_domain']


In [196]:
# Choose your feature columns
feature_cols = [
    'delta_hydrophobicity', 'delta_charge', 'delta_polarity',
    'delta_mw', 'delta_volume',
    'in_RING_domain', 'in_BRCT1_domain', 'in_BRCT2_domain'
]

X = df[feature_cols]
# Define your label (binary: 0 = benign, 1 = pathogenic)

#y = df['ClinicalSignificance'].map({'Benign': 0, 'Likely benign': 0, 'Benign/Likely benign':0, 'Pathogenic': 1, 'Likely pathogenic':1, 'Pathogenic/Likely pathogenic':1})

In [197]:
df['ClinicalSignificance'].value_counts() # allows to view the type of categories under clinical significance

ClinicalSignificance
Benign                          296
Pathogenic                      140
Pathogenic/Likely pathogenic    106
Benign/Likely benign             30
Name: count, dtype: int64

In [198]:
#we dont need these we we drop them from dataframe
to_drop = [
    'Conflicting classifications of pathogenicity',
    'Uncertain significance',
    'not provided'
]

df = df[~df['ClinicalSignificance'].isin(to_drop)].reset_index(drop=True)

In [199]:
# now we can map
y = df['ClinicalSignificance'].map({
    'Benign': 0, 
    'Likely benign': 0, 
    'Benign/Likely benign':0, 
    'Pathogenic': 1, 
    'Likely pathogenic':1, 
    'Pathogenic/Likely pathogenic':1
})
df = df[~y.isna()]
y = y.dropna()
# below we re-assign X to make sure the number of row in X matches the number of rows in y
feature_cols = [
    'delta_hydrophobicity', 'delta_charge', 'delta_polarity',
    'delta_mw', 'delta_volume',
    'in_RING_domain', 'in_BRCT1_domain', 'in_BRCT2_domain'
]

X = df[feature_cols]

In [200]:
from sklearn.model_selection import train_test_split
# 20 precent (test_size = 0.2) of data is for testing, while the 80 is for training the model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#print(len(X), len(y))

In [201]:
#training a bseline model - logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [202]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("logistic regression ROC AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.84      0.89      0.87        66
           1       0.84      0.78      0.81        49

    accuracy                           0.84       115
   macro avg       0.84      0.83      0.84       115
weighted avg       0.84      0.84      0.84       115

logistic regression ROC AUC: 0.8849721706864564


In [203]:
# now we try the random forest model
from sklearn.ensemble import RandomForestClassifier
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest_model.fit(X_train, y_train)

y_pred_forest = random_forest_model.predict(X_test)
y_proba_forest = random_forest_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_forest))
print("random forest ROC AUC:", roc_auc_score(y_test, y_proba_forest))

              precision    recall  f1-score   support

           0       0.91      0.95      0.93        66
           1       0.93      0.88      0.91        49

    accuracy                           0.92       115
   macro avg       0.92      0.92      0.92       115
weighted avg       0.92      0.92      0.92       115

random forest ROC AUC: 0.9768089053803339
